# Context Window Optimization — Exploration Notebook

Interactive exploration of the QA pipeline and context selection strategies.

**Run from the project root directory** so that `src/` imports work correctly:
```
cd /path/to/NLP_DL
jupyter notebook notebooks/exploration.ipynb
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))  # Add project root

import warnings
warnings.filterwarnings('ignore')

import yaml
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
print('Imports OK')

## 1. Load Config and Dataset

In [ ]:
with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

print(yaml.dump(cfg, default_flow_style=False))

In [ ]:
from src.data.qa_generator import build_synthetic_dataset, SYNTHETIC_STORIES
from src.data.dataset_loader import load_dataset

# Build dataset if needed
dataset_path = '../data/processed/dataset.json'
build_synthetic_dataset(dataset_path)
dataset = load_dataset(dataset_path)

print(f'Stories: {len(dataset)}')
print(f'Total QA pairs: {sum(len(d["qa_pairs"]) for d in dataset)}')

## 2. Chunking Exploration

In [ ]:
from src.utils.chunking import chunk_by_tokens, split_into_sentences

story = dataset[1]['story']  # Marie Curie
print('Story preview:', story[:200])
print()

chunks = chunk_by_tokens(story, chunk_size=100, overlap=15)
print(f'Chunks ({len(chunks)}):')
for i, c in enumerate(chunks):
    print(f'  [{i}] {c[:80]}...')

## 3. Embedding Model

In [ ]:
from src.models.embeddings import EmbeddingModel

emb = EmbeddingModel(cache_dir='../data/processed/embedding_cache')
q = 'What Nobel Prizes did Marie Curie win?'
q_vec = emb.encode(q)
print(f'Query embedding shape: {q_vec.shape}')

ranked = emb.rank_by_similarity(q, chunks)
print('\nTop chunks by similarity:')
for score, idx, text in ranked[:3]:
    print(f'  score={score:.3f} [{idx}]: {text[:80]}...')

## 4. Context Selectors

In [ ]:
from src.selectors.topk_selector import TopKSelector
from src.selectors.sliding_window import SlidingWindowSelector
from src.selectors.keyword_selector import KeywordSelector
from src.evaluation.evaluator import FullContextSelector, TruncatedSelector

question = 'What elements did Marie Curie discover?'

selectors = {
    'full': FullContextSelector(),
    'head': TruncatedSelector('head', 2),
    'topk': TopKSelector(emb, k=2),
    'sliding': SlidingWindowSelector(emb, window_size=2, stride=1, top_n=1),
    'keyword': KeywordSelector(num_keywords=8),
}

print(f'Question: {question}\n')
for name, sel in selectors.items():
    ctx, toks = sel.select(chunks, question)
    print(f'{name:12s} ({toks:4d} tokens): {ctx[:100]}...')

## 5. TinyLlama Inference (loads model — may take a moment)

In [ ]:
from src.models.tinyllama import TinyLlamaModel

llm = TinyLlamaModel(max_new_tokens=64, use_fp16=True)
print('Model ready.')

In [ ]:
# Test a single QA pair
qa = dataset[1]['qa_pairs'][2]  # polonium question
print(f'Question: {qa["question"]}')
print(f'Gold:     {qa["answer"]}')

topk_sel = TopKSelector(emb, k=2)
ctx, toks = topk_sel.select(chunks, qa['question'])
pred = llm.answer(ctx, qa['question'])

print(f'Pred:     {pred}')
print(f'Tokens used: {toks}')

## 6. Visualize Embedding Similarities

In [ ]:
questions = [qa['question'] for qa in dataset[1]['qa_pairs']]
q_vecs = emb.encode(questions)
c_vecs = emb.encode(chunks)

# Similarity heatmap: questions x chunks
sim_matrix = np.zeros((len(questions), len(chunks)))
for i, qv in enumerate(q_vecs):
    for j, cv in enumerate(c_vecs):
        sim_matrix[i, j] = emb.similarity(qv, cv)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(sim_matrix, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(chunks)))
ax.set_xticklabels([f'C{i}' for i in range(len(chunks))])
ax.set_yticks(range(len(questions)))
ax.set_yticklabels([q[:40] + '...' for q in questions], fontsize=8)
ax.set_title('Question-Chunk Similarity Heatmap (Marie Curie)')
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 7. Load and Visualize Experiment Results

In [ ]:
import glob

result_files = glob.glob('../results/*.json')
if not result_files:
    print('No results yet. Run experiments/run_baselines.py first.')
else:
    records = []
    for rf in result_files:
        with open(rf) as f:
            data = json.load(f)
        if isinstance(data, list):
            records.extend(data)
    
    df = pd.DataFrame(records)
    print(df[['method', 'accuracy', 'tokens_used', 'efficiency']].to_string())
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    methods = df.groupby('method')['accuracy'].mean()
    methods.sort_values(ascending=True).plot.barh(ax=ax1, color='steelblue')
    ax1.set_title('Accuracy by Method')
    ax1.set_xlabel('Substring Match Accuracy')
    
    toks = df.groupby('method')['tokens_used'].mean()
    toks.sort_values(ascending=True).plot.barh(ax=ax2, color='coral')
    ax2.set_title('Avg Token Usage by Method')
    ax2.set_xlabel('Tokens')
    
    plt.tight_layout()
    plt.show()